# 01 — Data Pipeline
Loads the SemEval 2014 Task 4 Restaurant datasets from local storage, parses both formats (XML for train, ASTE TXT for test), applies text standardisation, and saves clean CSVs for downstream use.

**Output files:**
- `../data/Train.csv` — parsed and standardised training set
- `../data/Test.csv` — parsed and standardised test set
- `../data/train_categorised.csv` — after aspect categorisation (produced at end of this notebook)
- `../data/test_categorised.csv` — after aspect categorisation

## 1. Imports

In [1]:
import os
import ast
import re
import html
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd
import spacy

pd.set_option('display.max_colwidth', None)

## 2. Load Local Data Files

In [2]:
data_dir = "../data"

train_path = os.path.join(data_dir, "SemEval14_Restaurants_Train.xml")
test_path  = os.path.join(data_dir, "SemEval14_Restaurants_Test.txt")

print(f"Train file found: {os.path.exists(train_path)}")
print(f"Test file found:  {os.path.exists(test_path)}")

Train file found: True
Test file found:  True


## 3. Dataset Parsing

### 3.1 Training Set (XML Format)

In [3]:
def parse_xml_format(file_path):
    """
    Parses SemEval 2014 XML format.
    Extracts text, aspect term, and polarity per annotation.
    Filters out 'conflict' polarity labels.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    data = []

    for sentence in root.findall('sentence'):
        text = sentence.find('text').text
        aspect_terms = sentence.find('aspectTerms')

        if aspect_terms is not None:
            for aspect in aspect_terms.findall('aspectTerm'):
                term     = aspect.get('term')
                polarity = aspect.get('polarity')

                # Filter out conflict labels — annotator disagreement, not usable
                if polarity in ['positive', 'negative', 'neutral']:
                    data.append({
                        'text':     text,
                        'aspect':   term,
                        'polarity': polarity
                    })

    return pd.DataFrame(data)


train_df = parse_xml_format(train_path)
print(f"Train rows extracted: {len(train_df)}")
train_df.head()

Train rows extracted: 3608


,text,aspect,polarity
0,But the staff was so horrible to us.,staff,negative
1,"To be completely fair, the only redeeming factor was the food, which was above average, but couldn't make up for all the other deficiencies of Teodora.",food,positive
2,"The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not.",food,positive
3,"The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not.",kitchen,positive
4,"The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not.",menu,neutral


### 3.2 Test Set (ASTE TXT Format)

In [4]:
def parse_aste_txt_format(file_path):
    """
    Parses ASTE triplet format: text####[(aspect_indices, opinion_indices, sentiment)]
    Reconstructs aspect term from word indices.
    Uses ast.literal_eval for safe parsing of the triplet list.
    """
    sentiment_map = {'POS': 'positive', 'NEG': 'negative', 'NEU': 'neutral'}
    data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if "####" not in line:
                continue

            parts    = line.split("####")
            text     = parts[0].strip()
            triplets = ast.literal_eval(parts[1].strip())
            words    = text.split()

            for triplet in triplets:
                aspect_indices  = triplet[0]
                sentiment_label = triplet[2]
                aspect_text     = " ".join([words[i] for i in aspect_indices])

                data.append({
                    'text':     text,
                    'aspect':   aspect_text,
                    'polarity': sentiment_map.get(sentiment_label, 'neutral')
                })

    return pd.DataFrame(data)


test_df = parse_aste_txt_format(test_path)
print(f"Test rows extracted: {len(test_df)}")
test_df.head()

Test rows extracted: 994


,text,aspect,polarity
0,The bread is top notch as well .,bread,positive
1,I have to say they have one of the fastest delivery times in the city .,delivery times,positive
2,Food is always fresh and hot ready to eat !,Food,positive
3,Food is always fresh and hot ready to eat !,Food,positive
4,Did I mention that the coffee is OUTSTANDING ?,coffee,positive


## 4. Quick Null Check

In [5]:
print("=== NULL CHECK — TRAIN ===")
print(train_df.isnull().sum())

print("\n=== NULL CHECK — TEST ===")
print(test_df.isnull().sum())

=== NULL CHECK — TRAIN ===
text        0
aspect      0
polarity    0
dtype: int64

=== NULL CHECK — TEST ===
text        0
aspect      0
polarity    0
dtype: int64


## 5. Text Standardisation

Cleans raw text before any NLP processing. Applied to both the review text and the aspect term.

Steps: HTML entity decoding → unicode normalisation → lowercasing → whitespace normalisation.

In [6]:
def standardize_text(text):
    """
    Standardises raw text for downstream NLP processing.
    Does NOT remove stop-words or lemmatize — those steps happen
    immediately before vectorisation in the model training pipeline.
    """
    if not isinstance(text, str):
        return ""

    # 1. Decode HTML entities (e.g. &amp; → &)
    text = html.unescape(text)

    # 2. Normalise unicode (e.g. smart quotes → standard quotes)
    text = unicodedata.normalize('NFKC', text)

    # 3. Lowercase
    text = text.lower()

    # 4. Collapse multiple whitespace into single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply to both text and aspect columns
for df in [train_df, test_df]:
    df['text_clean']   = df['text'].apply(standardize_text)
    df['aspect_clean'] = df['aspect'].apply(standardize_text)

print("Standardisation applied.")
print(f"Train columns: {list(train_df.columns)}")
print(f"Test columns:  {list(test_df.columns)}")

Standardisation applied.
Train columns: ['text', 'aspect', 'polarity', 'text_clean', 'aspect_clean']
Test columns:  ['text', 'aspect', 'polarity', 'text_clean', 'aspect_clean']


## 6. Save Clean CSVs

In [7]:
train_df.to_csv(os.path.join(data_dir, "Train.csv"), index=False)
test_df.to_csv(os.path.join(data_dir, "Test.csv"),   index=False)

print(f"Train.csv saved — {len(train_df)} rows, columns: {list(train_df.columns)}")
print(f"Test.csv saved  — {len(test_df)} rows, columns: {list(test_df.columns)}")

Train.csv saved — 3608 rows, columns: ['text', 'aspect', 'polarity', 'text_clean', 'aspect_clean']
Test.csv saved  — 994 rows, columns: ['text', 'aspect', 'polarity', 'text_clean', 'aspect_clean']


## 7. Aspect Categorisation — Generate Categorised CSVs

Maps `aspect_clean` to one of five macro-categories using `AspectCategoriser` from `src/categoriser.py`.

These files are required by the model training notebook which uses `macro_aspect` as a one-hot encoded feature.

**Output files:**
- `../data/train_categorized.csv`
- `../data/test_categorized.csv`

In [8]:
import sys
sys.path.append('../src')

from categoriser import AspectCategoriser
from utils import standardize_dataframe_aspects

# Load spaCy if not already loaded
import spacy
nlp = spacy.load('en_core_web_md')

# Instantiate categoriser — use 'drop' for training data (clean labels only)
categoriser = AspectCategoriser(nlp_model=nlp, on_no_match='drop')

print('Categoriser ready.')

Categoriser ready.


In [9]:
print('Categorising training data...')
train_clean = standardize_dataframe_aspects(train_df, categoriser, label='Train')

print('\nCategorising test data...')
test_clean = standardize_dataframe_aspects(test_df, categoriser, label='Test')

print(f'\nFinal train shape: {train_clean.shape}')
print(f'Final test shape:  {test_clean.shape}')
print(f'Columns: {list(train_clean.columns)}')

Categorising training data...
Train: 3608 rows → dropped 547 (15.2%) → kept 3061 (84.8%)

Categorising test data...
Test: 994 rows → dropped 92 (9.3%) → kept 902 (90.7%)

Final train shape: (3061, 6)
Final test shape:  (902, 6)
Columns: ['text', 'aspect', 'polarity', 'text_clean', 'aspect_clean', 'macro_aspect']


In [10]:
train_clean.to_csv('../data/train_categorized.csv', index=False)
test_clean.to_csv('../data/test_categorized.csv',   index=False)

print('Saved:')
print('  ../data/train_categorized.csv')
print('  ../data/test_categorized.csv')

Saved:
  ../data/train_categorized.csv
  ../data/test_categorized.csv
